Voici le corrigé complet, rédigé selon les standards d'une **revue d'architecture en entreprise**, pour l'atelier de conception d'une infrastructure d'agent de conformité bancaire.

---

# Corrigé de l'Atelier : Conception d'une Architecture de Conformité Réglementaire Bancaire

## 1. Schéma d'Architecture Cible (VPC, Subnets & Ségrégation)

Le schéma ci-dessous modélise la ségrégation réseau stricte requise pour protéger les données financières et isoler les outils d'exécution :

```mermaid
graph TD
    subgraph Public_Internet [Internet Public]
        Client[Navigateur / App Client]
    end

    subgraph VPC_Secure
        subgraph DMZ
            Gateway
        end

        subgraph App_Subnet
            FastAPI
            Redis
        end

        subgraph Data_Subnet
            Postgres
        end

        subgraph Integration_Subnet
            MCPServer
        end
    end

    subgraph External_Cloud [Inférence Cloud Privée]
        LLM[API LLM Externe - ex: Azure OpenAI Private Link]
    end

    Client -->|HTTPS| Gateway
    Gateway -->|Routage interne HTTP/SSE| FastAPI
    FastAPI <-->|Redis Protocol TCP| Redis
    FastAPI <-->|SQL TCP 5432| Postgres
    FastAPI <-->|JSON-RPC via stdio/SSE| MCPServer
    FastAPI <-->|HTTPS Private Link| LLM

```

### Justification de la ségrégation réseau :

* **DMZ (API Gateway)** : Seul point d'entrée exposé à l'internet public. L'API Gateway gère l'authentification mutuelle (mTLS) et le chiffrement HTTPS.
* **App Subnet (FastAPI & Redis)** : Sous-réseau privé sans adresse IP publique. Il héberge les serveurs FastAPI stateless et le cache Redis.
* **Data Subnet (PostgreSQL)** : Complètement isolé de l'applicatif par des règles de groupe de sécurité strictes. Seul le port 5432 provenant de l'App Subnet y est autorisé.
* **Integration Subnet (Serveurs MCP)** : Les connecteurs réseau (APIs de crédit, parsers de documents contractuels PDF) sont isolés dans un sous-réseau dédié afin de limiter le rayon d'impact en cas d'injection de commande ou de compromission d'un outil tiers.

---

## 2. Cycle de vie d'une requête d'audit (Data Flow)

Voici le parcours d'exécution d'une requête utilisateur : « Analyse le contrat de prêt #432 et vérifie s'il respecte le taux d'usure légal » :

1. **Réception et Contrôle (API Gateway)** : La requête HTTPS arrive sur la Gateway. Elle valide le jeton JWT de l'utilisateur, applique un filtre de limitation de débit (Rate Limiting) et transmet la requête propre à FastAPI.
2. **Réhydratation de Session (FastAPI)** : FastAPI extrait l'identifiant de session (`session_id`). Il interroge la base Redis (cache chaud) pour récupérer le dictionnaire JSON de l'historique conversationnel de cette session.
3. **Acquisition du Verrou (Redis)** : Avant d'instancier l'agent, FastAPI acquiert un verrou distribué sur Redis pour bloquer toute autre requête concurrente sur cette session (voir section 3).
4. **Instanciation de l'Agent** : L'agent éphémère est instancié à la volée en lui injectant l'historique conversationnel réhydraté, sa configuration et le registre d'outils.
5. **Première Inférence (LLM)** : L'agent envoie l'historique enrichi du nouveau prompt utilisateur au LLM via une liaison privée (Private Link). Le modèle formule une demande d'outil (Function Calling) : `interroger_api_credit(pret_id=432)`.
6. **Exécution de l'Outil (Serveur MCP)** : L'agent intercepte la demande d'outil et transmet la commande en format JSON-RPC sécurisé au serveur d'outils MCP isolé. L'API de crédit renvoie les données financières du prêt.
7. **Seconde Inférence et Synthèse (LLM)** : L'observation financière est réinjectée dans le fil conversationnel. L'agent effectue un second appel LLM. Le modèle formule sa conclusion d'audit finale.
8. **Persistance de l'État (Redis / Postgres)** : Une fois la boucle ReAct terminée, FastAPI met à jour l'historique conversationnel dans Redis (TTL de 2 heures) et écrit de manière asynchrone un enregistrement d'audit persistant dans PostgreSQL. Le verrou Redis est immédiatement relâché.
9. **Streaming de Sortie (Client)** : FastAPI diffuse la réponse finale au client en utilisant un flux continu Server-Sent Events (SSE) pour une réactivité optimale.

---

## 3. Stratégie de prévention des corruptions de session (Appels concurrents)

Dans un environnement bancaire, deux utilisateurs (ou deux processus asynchrones) ne doivent pas pouvoir altérer simultanément l'état d'une même session. Si cela se produisait, l'ordre logique des messages `$S_{t+1}=\Phi(S_t,A_t)$` serait corrompu, provoquant des plantages de l'API de complétion et des hallucinations critiques d'outils.

Pour neutraliser ce risque, nous implémentons le pattern du **Verrou Distribué avec Redis (Principe Redlock)** :

### Algorithme d'acquisition du verrou par l'API FastAPI :

1. **Tentative d'écriture exclusive** : Dès la réception de la requête, FastAPI tente d'écrire une clé de verrou unique dans Redis à l'aide de la commande atomique `SET` :
```bash
SET lock:session_432 unique_request_token NX PX 30000

```


* `NX` : Écrit la clé uniquement si elle n'existe pas déjà.
* `PX 30000` : Configure une expiration automatique (TTL) de 30 secondes pour éviter un blocage infini du système en cas de plantage d'un serveur applicatif.


2. **Gestion de la concurrence** :
* **Si la commande réussit (retourne `OK`)** : FastAPI détient le verrou exclusif. Il peut procéder en toute sécurité à la réhydratation de l'agent et à l'exécution de la boucle ReAct.
* **Si la commande échoue (retourne `nil`)** : Une requête est déjà en cours de traitement sur cette session. FastAPI rejette immédiatement l'appel entrant en retournant un code d'erreur standardisé `409 Conflict` (ou place la tâche dans une file d'attente asynchrone Redis Celery selon la logique métier retenue).


3. **Libération sécurisée du verrou** : Une fois la persistance finale effectuée dans Redis et PostgreSQL, l'instance FastAPI libère le verrou en exécutant un script Lua atomique sur Redis :
```lua
if redis.call("get", KEYS[1]) == ARGV[1] then
    return redis.call("del", KEYS[1])
else
    return 0
end

```


*Ce script Lua s'assure que le serveur applicatif ne libère que le verrou qu'il a lui-même acquis (en comparant la valeur de la clé au `unique_request_token` initial), évitant qu'un worker lent ne supprime accidentellement le verrou d'un autre worker ayant démarré entre-temps.*